# Application de K-Means et segmentation en 4 profils

Ce notebook entraîne le modèle K-Means final avec K=4, profile chaque cluster, et assigne l'un des 4 segments business demandés : **Champions**, **Réguliers**, **À Surveiller**, **Perdus**.

## Import des librairies et configuration des chemins

In [1]:
from pathlib import Path  # Construire des chemins robustes.
import os  # Créer les dossiers de sortie si besoin.
import pickle  # Sauvegarder le modèle entraîné.
import pandas as pd  # Traiter les tables de clustering.
from sklearn.cluster import KMeans  # Entraîner le modèle K-Means final.

PROJECT_ROOT = Path('.')  # Racine du projet.
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'  # Dossier des données préparées.
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'  # Dossier des résultats finaux.
MODELS_DIR = PROJECT_ROOT / 'models'  # Dossier des modèles.
os.makedirs(RESULTS_DIR, exist_ok=True)  # S'assurer que le dossier results existe.
os.makedirs(MODELS_DIR, exist_ok=True)  # S'assurer que le dossier models existe.

## Charger les données préparées

In [2]:
X_scaled_df = pd.read_csv(PROCESSED_DIR / 'X_scaled.csv')  # Charger les features RFM standardisées.
rfm = pd.read_csv(PROCESSED_DIR / 'rfm_raw_table.csv')  # Charger la table RFM brute par client (avec ClientID).

K = 4  # Nombre de clusters fixé pour correspondre aux 4 segments business demandés.

## Entraîner K-Means et assigner les clusters

In [3]:
kmeans = KMeans(n_clusters=K, init='k-means++', random_state=42, n_init=10)  # Initialiser le modèle final.
rfm['cluster'] = kmeans.fit_predict(X_scaled_df)  # Entraîner et assigner les clusters.
print('Répartition des clients par cluster :')  # Titre de la répartition.
print(rfm['cluster'].value_counts().sort_index())  # Afficher le nombre de clients par cluster.

Répartition des clients par cluster :
cluster
0    2010
1     440
2    1575
3     913
Name: count, dtype: int64


## Profiler les clusters et assigner les 4 segments

Chaque cluster est classé selon un score combiné : rang de récence (plus bas = plus récent), rang de fréquence (plus bas = plus fréquent), rang de montant (plus bas = plus dépensier). Le score total (somme des 3 rangs) ordonne les clusters du meilleur au moins bon.

In [4]:
cluster_summary = rfm.groupby('cluster')[['Recency', 'Frequency', 'Monetary']].mean()  # Profil moyen par cluster.
print(cluster_summary)  # Afficher le profil des clusters.

rec_rank = cluster_summary['Recency'].rank(ascending=True, method='min')  # Rang récence : 1 = client le plus récent.
freq_rank = cluster_summary['Frequency'].rank(ascending=False, method='min')  # Rang fréquence : 1 = client le plus fréquent.
mon_rank = cluster_summary['Monetary'].rank(ascending=False, method='min')  # Rang montant : 1 = client qui dépense le plus.
combined_score = rec_rank + freq_rank + mon_rank  # Score combiné : plus bas = meilleur profil client.
ordered_clusters = combined_score.sort_values().index.tolist()  # Trier les clusters du meilleur au moins bon.
print('\nScore combiné par cluster (plus bas = meilleur) :')  # Titre du score combiné.
print(combined_score.sort_values())  # Afficher le score combiné trié.

segment_names = ['Champions', 'Réguliers', 'À Surveiller', 'Perdus']  # Ordre business du meilleur au moins bon.
cluster_labels = {int(cid): segment_names[i] for i, cid in enumerate(ordered_clusters)}  # Associer chaque cluster à son segment.
print('\nCorrespondance cluster -> segment :')  # Titre de la correspondance.
print(cluster_labels)  # Afficher la correspondance finale.

rfm['SegmentRFM'] = rfm['cluster'].map(cluster_labels)  # Ajouter la colonne de segment final.

            Recency  Frequency       Monetary
cluster                                      
0        165.030348   3.636816   30128.596174
1        162.861364   7.145455  199213.968745
2        147.934603   7.336508   49412.502488
3        621.646221   3.276013   28353.731209

Score combiné par cluster (plus bas = meilleur) :
cluster
2     4.0
1     5.0
0     9.0
3    12.0
dtype: float64

Correspondance cluster -> segment :
{2: 'Champions', 1: 'Réguliers', 0: 'À Surveiller', 3: 'Perdus'}


## Ajouter les actions recommandées

In [5]:
action_map = {  # Définir l'action recommandée pour chaque segment.
    'Champions': 'Programme VIP, récompenses exclusives, vente croisée',
    'Réguliers': 'Offres de fidélisation, points bonus, upselling',
    'À Surveiller': 'Relance légère, offre personnalisée, sondage satisfaction',
    'Perdus': 'Campagne de réactivation, offre de retour spéciale'
}  # Correspondance segment -> action.
rfm['ActionRecommandee'] = rfm['SegmentRFM'].map(action_map)  # Ajouter la colonne d'action recommandée.
print(rfm['SegmentRFM'].value_counts())  # Afficher la distribution finale des segments.
print('\nProfil moyen par segment :')  # Titre du profil par segment.
print(rfm.groupby('SegmentRFM')[['Recency', 'Frequency', 'Monetary']].mean().round(1))  # Afficher le profil moyen par segment.

SegmentRFM
À Surveiller    2010
Champions       1575
Perdus           913
Réguliers        440
Name: count, dtype: int64

Profil moyen par segment :
              Recency  Frequency  Monetary
SegmentRFM                                
Champions       147.9        7.3   49412.5
Perdus          621.6        3.3   28353.7
Réguliers       162.9        7.1  199214.0
À Surveiller    165.0        3.6   30128.6


## Sauvegarder le modèle et le fichier segmenté final

In [6]:
with open(MODELS_DIR / 'kmeans_model.pkl', 'wb') as f:  # Ouvrir le fichier modèle en écriture binaire.
    pickle.dump(kmeans, f)  # Sauvegarder le modèle K-Means entraîné.
rfm.to_csv(RESULTS_DIR / 'rfm_with_clusters.csv', index=False)  # Sauvegarder la table finale segmentée.
print(f"Sauvegardé : {MODELS_DIR / 'kmeans_model.pkl'}")  # Confirmer le chemin du modèle.
print(f"Sauvegardé : {RESULTS_DIR / 'rfm_with_clusters.csv'}")  # Confirmer le chemin du fichier final.

Sauvegardé : models/kmeans_model.pkl
Sauvegardé : data/results/rfm_with_clusters.csv
